# Fine-tuning RF-DETR-S sur American Sign Language Letters

**Plateforme :** Google Colab (Runtime > Change runtime type > T4 GPU)

**Objectif :** Entraîner un modèle RF-DETR-S sur le dataset ASL en parallèle de YOLO pour la comparaison.

**Durée estimée :** ~2-3h sur T4 (30 epochs, batch_size=4, grad_accum_steps=4).

## 1. Vérification GPU

In [ ]:
!nvidia-smi

## 2. Installation des dépendances

In [ ]:
!pip install -q "rfdetr[train,loggers]" roboflow supervision

## 3. Téléchargement du dataset ASL au format COCO

In [ ]:
import os
from roboflow import Roboflow

API_KEY = os.environ.get("ROBOFLOW_API_KEY", "01ppKST0VVzBcfnAElsw")

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("david-lee-d0rhs").project("american-sign-language-letters")
version = project.version(6)
dataset = version.download("coco")

print("Dataset téléchargé dans:", dataset.location)
!ls {dataset.location}

In [ ]:
# RF-DETR attend les sous-dossiers train/, valid/, test/ avec _annotations.coco.json
# Roboflow exporte exactement ce format avec download("coco")
import json
from pathlib import Path

train_ann = Path(dataset.location) / "train" / "_annotations.coco.json"
with open(train_ann) as f:
    coco = json.load(f)
print(f"Nombre de classes: {len(coco['categories'])}")
print(f"Classes: {[c['name'] for c in coco['categories']]}")
print(f"Nombre d'images train: {len(coco['images'])}")
print(f"Nombre d'annotations train: {len(coco['annotations'])}")

## 4. Entraînement RF-DETR-S

- 30 epochs (RF-DETR converge plus vite que YOLO sur petit dataset)
- batch_size=4, grad_accum_steps=4 (effective batch=16, T4-friendly)
- Sauvegarde tous les 5 epochs

In [ ]:
from rfdetr import RFDETRSmall
import torch

torch.manual_seed(42)

model = RFDETRSmall()

model.train(
    dataset_dir=dataset.location,
    epochs=30,
    batch_size=4,
    grad_accum_steps=4,
    lr=1e-4,
    output_dir='./output_rfdetr_asl',
    early_stopping=True,
    early_stopping_patience=10,
    seed=42,
)

## 5. Évaluation sur le test set

In [ ]:
from rfdetr import RFDETRSmall
import os, glob

# Recherche du best checkpoint (recursive si besoin)
candidates = sorted(glob.glob('**/checkpoint_best_total.pth', recursive=True),
                    key=os.path.getmtime, reverse=True)
if not candidates:
    candidates = sorted(glob.glob('**/checkpoint*.pth', recursive=True),
                        key=os.path.getmtime, reverse=True)

assert candidates, "Aucun checkpoint trouvé — le training a-t-il bien terminé ?"
best_ckpt = candidates[0]
print(f"Utilisation de: {best_ckpt}")

best_model = RFDETRSmall(pretrain_weights=best_ckpt)

# Évaluation - l'API peut varier selon la version de rfdetr.
# Si .evaluate() ne marche pas, on fera l'éval dans le notebook 04 via supervision.
try:
    from pathlib import Path
    metrics = best_model.evaluate(dataset_dir=dataset.location, split='test')
    print('Métriques:', metrics)
except Exception as e:
    print(f"[INFO] .evaluate() non disponible ou a échoué ({e}).")
    print("L'évaluation comparative sera réalisée dans le notebook 04 via supervision.")

## 6. Temps d'inférence

In [ ]:
import time
import glob
from PIL import Image

test_images = sorted(glob.glob(f"{dataset.location}/test/*.jpg"))[:50]
print(f"Mesure sur {len(test_images)} images")

img0 = Image.open(test_images[0])
_ = best_model.predict(img0)  # warmup

start = time.time()
for img_path in test_images:
    img = Image.open(img_path)
    _ = best_model.predict(img)
elapsed = time.time() - start
print(f"Temps moyen par image: {1000*elapsed/len(test_images):.2f} ms")
print(f"FPS estimé: {len(test_images)/elapsed:.1f}")

## 7. Sauvegarde et téléchargement

In [ ]:
import shutil, os
from google.colab import files

size_mb = os.path.getsize(best_ckpt) / 1024 / 1024
print(f"Taille modèle best: {size_mb:.2f} MB")

# Archive du dossier de sortie (où qu'il soit)
output_dir = os.path.dirname(best_ckpt)
shutil.make_archive('rfdetr_asl_run', 'zip', output_dir)
print(f"Archive créée: rfdetr_asl_run.zip (depuis {output_dir})")

files.download(best_ckpt)
files.download('rfdetr_asl_run.zip')